In [11]:
import json
import numpy as np
from scipy import stats
from itertools import combinations
import pandas as pd
import mlflow
import os

BASE_DIR = os.path.abspath("..")
mlflow.set_tracking_uri(f"sqlite:///{BASE_DIR}/mlflow.db")
EXPERIMENT_NAME = "NACC - finalne modely s kalibráciou"
TARGET_RUNS = {"LogReg_FINAL_calibrated_MODEL", "SVM_FINAL_calibrated_MODEL", "XGBoost_FINAL_calibrated_MODEL"}

### DeLong test - je rozdiel medzi ROC AUC hodnotami modelov štatisticky významný?

In [12]:
def load_predictions(experiment_name):
    client = mlflow.tracking.MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' nebol nájdený")

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="attributes.status = 'FINISHED'"
    )

    model_probabilities = {}
    y_true = None

    for run in runs:
        run_name = run.data.tags.get("mlflow.runName", "")
        if run_name not in TARGET_RUNS:
            continue

        model_name = run.data.params.get("model")
        artifact_path = client.download_artifacts(run.info.run_id, "predictions.json")
        with open(artifact_path, "r") as f:
            data = json.load(f)

        model_probabilities[model_name] = np.array(data["y_prob"])
        if y_true is None:
            y_true = np.array(data["y_true"])

        print(f"Načítaný model: {model_name}")
    return y_true, model_probabilities


y_true, models_probabilities = load_predictions(EXPERIMENT_NAME)

Načítaný model: xgboost


Načítaný model: svm


Načítaný model: lr


In [13]:
def compute_auc(y_true, y_predicted):
    pos_scores = y_predicted[y_true == 1]
    neg_scores = y_predicted[y_true == 0]
    positive_pocet = len(pos_scores)
    negative_pocet = len(neg_scores)

    V10 = np.array([np.mean(p > neg_scores) + 0.5 * np.mean(p == neg_scores) for p in pos_scores]) # kolko negativnych predikcii porazi kazda pozitivna predikcia
    V01 = np.array([np.mean(q < pos_scores) + 0.5 * np.mean(q == pos_scores) for q in neg_scores]) # kolko pozitivnych predikcii porazi kazdu negativnu predikciu

    auc = np.mean(V10)
    variance = np.var(V10, ddof=1) / positive_pocet + np.var(V01, ddof=1) / negative_pocet
    return auc, variance, V10, V01


def delong_test(y_true, y_predicted_1, y_predicted_2):
    pos_idx = y_true == 1
    neg_idx = y_true == 0
    positive_pocet = pos_idx.sum()
    negative_pocet = neg_idx.sum()
    auc1, var1, V10_1, V01_1 = compute_auc(y_true, y_predicted_1)
    auc2, var2, V10_2, V01_2 = compute_auc(y_true, y_predicted_2)
    cov = (np.cov(V10_1, V10_2)[0, 1] / positive_pocet) + (np.cov(V01_1, V01_2)[0, 1] / negative_pocet)
    std_dev = np.sqrt(var1 + var2 - 2 * cov)
    z = (auc1 - auc2) / std_dev             
    p = 2 * (1 - stats.norm.cdf(abs(z)))

    return {
        "auc1": auc1,
        "auc2": auc2,
        "auc_diff": auc1 - auc2,
        "z_statistic": z,
        "p_value": p,
        "significant": p < 0.05
    }

In [14]:
model_names = list(models_probabilities.keys())
pairs = list(combinations(model_names, 2))
results = []

print("DeLong test – porovnanie ROC AUC modelov")
print("=" * 65)
for name1, name2 in pairs:
    vysledok = delong_test(y_true, models_probabilities[name1], models_probabilities[name2])
    significance = "VÝZNAMNÝ ROZDIEL"  if vysledok["significant"] else "NEvýznamný rozdiel"

    print(f"\n{name1} vs {name2}")
    print(f"  AUC {name1}:   {vysledok['auc1']:.4f}")
    print(f"  AUC {name2}:   {vysledok['auc2']:.4f}")
    print(f"  Rozdiel:         {vysledok['auc_diff']:+.4f}")
    print(f"  z-štatistika:    {vysledok['z_statistic']:.4f}")
    print(f"  p-hodnota:       {vysledok['p_value']:.4f}")
    print(f"  Výsledok:        {significance}  (α = 0.05)")

    results.append({
        "Model 1": name1,
        "Model 2": name2,
        "AUC (Model 1)": round(vysledok["auc1"], 4),
        "AUC (Model 2)": round(vysledok["auc2"], 4),
        "Rozdiel AUC": round(vysledok["auc_diff"], 4),
        "z-štatistika": round(vysledok["z_statistic"], 4),
        "p-hodnota": round(vysledok["p_value"], 4),
        "Štatisticky významný rozdiel (p<0.05)": vysledok["significant"]
    })

DeLong test – porovnanie ROC AUC modelov

xgboost vs svm
  AUC xgboost:   0.7993
  AUC svm:   0.7940
  Rozdiel:         +0.0053
  z-štatistika:    1.3385
  p-hodnota:       0.1807
  Výsledok:        NEvýznamný rozdiel  (α = 0.05)

xgboost vs lr
  AUC xgboost:   0.7993
  AUC lr:   0.7957
  Rozdiel:         +0.0036
  z-štatistika:    0.8082
  p-hodnota:       0.4190
  Výsledok:        NEvýznamný rozdiel  (α = 0.05)

svm vs lr
  AUC svm:   0.7940
  AUC lr:   0.7957
  Rozdiel:         -0.0017
  z-štatistika:    -0.3790
  p-hodnota:       0.7047
  Výsledok:        NEvýznamný rozdiel  (α = 0.05)


In [15]:
pd.DataFrame(results)

,Model 1,Model 2,AUC (Model 1),AUC (Model 2),Rozdiel AUC,z-štatistika,p-hodnota,Štatisticky významný rozdiel (p<0.05)
0,xgboost,svm,0.7993,0.7940,0.0053,1.3385,0.1807,False
1,xgboost,lr,0.7993,0.7957,0.0036,0.8082,0.4190,False
2,svm,lr,0.7940,0.7957,-0.0017,-0.3790,0.7047,False
